# Geração de Sequência de Proteínas com Modelo de Transformadores

In [1]:
# Imports
import time
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

In [2]:
# Define o device
device = torch.device("cpu" if torch.backends.mps.is_built() else "cuda")
print("Device:", device)

Device: cuda


## Dados

Usaremos dados de sequências de proteínas, onde cada proteína é descrita por uma sequência de  Aminoácidos (AA).
Cada  aminoácido  exclusivo  é  representado  por  uma  letra exclusiva do alfabeto. 

Veja aqui o que cada letra representa: 

https://www.bioinformatics.org/sms/iupac.html

Exemplo do formato dos dados: MAFLKKSLFLVLFLGVVSLSFCEEEKREEHEEEKRDEEDAESLGKR

As  proteínas  são  de  eucariontes,  filtradas  para  ter comprimento de 40-200 AA, e foram originalmente retiradas do site UniProt: https://www.uniprot.org

### Carrega dados

In [6]:
def carrega_dados_json(filename):
    
    with open(filename) as f:
        
        data = json.load(f)

    return data

In [7]:
sequences = carrega_dados_json("dados.json")

In [8]:
print(f"Número de sequências de proteínas: {len(sequences)}")

Número de sequências de proteínas: 42980


In [15]:
sequences[30:34]

['MRTLWIMAVLLLGVEGDLMQFETLIMKIAKRSGMFWYSAYGCYCGWGGQGRPQDATDRCCFVHDCCYGKVTGCDPKLDSYTYSVENGDVVCGGNDPCKKEICECDRAAAICFRDNKVTYDNKYWRFPPQNCKEESEPC',
 'MKLLAPVLWAMAALGVTWLVAVDSKESCTKHSNGCSTPLRLPCQEYFRPACDIHDNCYHCGTIFGISRKECDDAFLKDMNTLCKKLGSNSATCPARGKREVTSHRATSIAHSRLWKTALDQKSFLNRKARQAILLTPNSCLYWANNFYMAVHVFGARSYSRTTDPKDCQGLKHCLPNH',
 'MKFIIVLLLLTALTLTSIPVIEGILKRCKTYDDCKDVCKARKGKCEFGICKCMIKSGK',
 'MLLYICLVNLLLPLSVGAASGAALGVIAKVGVDAALQQIDDVWKGKTVRYWKCAVENRSSKTLYALGTTQESGSMTTVFADIPPKSTGVFVWEKSRGAAKGAVGVVHYKYGNKVLNIMASIPYDWNLYKAWANVHLSDHKESFSDLYKGKNGAKYPTRAGNWGEVDGTKFFLTEKSHAEFKVIFSG']

### Exploração dos dados

In [18]:
# Verifica o tamanho (len) de cada sequência
seq_lens = [len(seq) for seq in sequences]

In [19]:
print(f"Comprimento Mínimo: {min(seq_lens)}")
print(f"Comprimento Médio: {sum(seq_lens) / len(seq_lens)}")
print(f"Comprimento Máximo: {max(seq_lens)}")

Comprimento Mínimo: 40
Comprimento Médio: 125.96361098185203
Comprimento Máximo: 200


In [23]:
# Função que recebe uma lista de sequências como argumento
def get_token_freq(sequences):
    
    # Inicializa um dicionário vazio para armazenar a frequência dos tokens
    token_freq = {}
    
    # Itera sobre cada sequência
    for seq in sequences:
        
        # Itera sobre cada token na sequência atual
        for token in seq:
            
            if token not in token_freq:
                
                # Adiciona o token ao dicionário com uma contagem inicial de 0
                token_freq[token] = 0
            
            # Incrementa a contagem de frequência do token em 1
            token_freq[token] += 1
    
    # Retorna o dicionário contendo as frequências dos tokens
    return token_freq

In [ ]:
aa_freq = get_token_freq(sequences);aa_freq

Ordena as chaves do dicionário freq com base nos valores associados a essas chaves, do maior para o menor

In [47]:
aa_sorted = sorted(aa_freq.keys(), key = lambda x: aa_freq[x], reverse = True)  

In [48]:
print(f"Número de aminoácidos diferentes: {len(aa_freq)}")
print(f"Número total de aminoácidos: {sum(aa_freq.values()):,}")

Número de aminoácidos diferentes: 24
Número total de aminoácidos: 5,413,916


In [ ]:
# Plot
plt.figure(figsize = (12, 5))
plt.title("Frequência de Aminoácidos")
plt.bar(range(len(aa_freq)), 
        [aa_freq[aa] for aa in aa_sorted], 
        align = "center", 
        tick_label = aa_sorted, 
        color = 'pink')
plt.xlabel("Aminoácidos")
plt.ylabel("Frequência")
plt.show()

In [51]:
print(aa_sorted)

['L', 'S', 'A', 'K', 'G', 'V', 'E', 'R', 'T', 'I', 'P', 'D', 'F', 'N', 'Q', 'Y', 'C', 'M', 'H', 'W', 'X', 'Z', 'B', 'U']
